# FINS experiment suite — Colab runner

Runs the FINS experiment scripts on a Colab GPU.

**What you need to upload:** a zip of the repository (`pde/` and `experiments/`).
A zip of `experiments/` alone is *not* enough — every script imports from
`../pde`, so the repository tree has to be present. If you only have the
experiments zip, the setup cell will clone the repository from GitHub and drop
the experiments folder inside it.

**Runtime:** `Runtime -> Change runtime type -> T4 GPU`. Experiments 1-4 and 8
are pure NumPy on CPU and are unaffected by the GPU choice; 5, 6, 7, 9, 10
train surrogates and want the GPU.

---

| script | what it answers | device |
|---|---|---|
| exp1 | do the tabulated ground-truth algebras hold on the torus? | CPU |
| exp2 | what does a random trivial-subalgebra guess score? | CPU |
| exp3 | how does the residual depend on domain geometry L/w? | CPU |
| exp4 | does the residual track a controlled symmetry breaking? | CPU |
| exp5 | how does the residual depend on surrogate quality? | GPU |
| exp6 | does the phi = a*u + b ansatz remove the trivial subalgebra? | GPU |
| exp7 | is the generator parameterisation local? | GPU |
| exp8 | does a seam-excluding window recover the full algebra? | CPU |
| exp9 | does the surrogate Jacobian respect the domain of dependence? | GPU |
| exp10 | does the windowed test survive on a surrogate? | GPU |

## 1. Environment

In [ ]:
import os, sys, subprocess, textwrap
import torch

print('python  :', sys.version.split()[0])
print('torch   :', torch.__version__)
print('cuda    :', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')
if not torch.cuda.is_available():
    print('\nNo GPU. Experiments 1-4 and 8 still run at full scale;',
          'the surrogate experiments will be slow — use --smoke for those.')

### Optional: mount Google Drive

Colab sessions disconnect; the full pipeline below runs for hours. With
`USE_DRIVE = True` the results and checkpoints are mirrored to Drive after
each section, so a disconnect costs a run, not the campaign.

In [ ]:
USE_DRIVE = False
DRIVE_DIR = '/content/drive/MyDrive/fins_runs'

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs(DRIVE_DIR, exist_ok=True)
    # every experiment script writes through exp_common.RESULTS_DIR,
    # which honours this variable — artefacts land on Drive directly
    os.environ['FINS_RESULTS_DIR'] = os.path.join(DRIVE_DIR, 'results')
    os.makedirs(os.environ['FINS_RESULTS_DIR'], exist_ok=True)
    print('results ->', os.environ['FINS_RESULTS_DIR'])

def mirror_to_drive(*paths):
    if not USE_DRIVE:
        return
    import shutil as _sh
    for p in paths:
        if os.path.isdir(p):
            _sh.copytree(p, os.path.join(DRIVE_DIR, os.path.basename(p)),
                         dirs_exist_ok=True)
        elif os.path.exists(p):
            _sh.copy(p, DRIVE_DIR)
    print('mirrored to', DRIVE_DIR)

## 2. Upload and unpack

Run the cell, pick the repository zip in the file dialog. If the file is
already sitting in `/content` (or on Drive), set `ZIP_PATH` and skip the
dialog.

In [ ]:
ZIP_PATH = None          # e.g. '/content/fins_repo_full.zip' to skip the dialog

if ZIP_PATH is None:
    from google.colab import files
    uploaded = files.upload()
    ZIP_PATH = '/content/' + next(iter(uploaded))
print('archive:', ZIP_PATH)

import zipfile, glob, shutil

EXTRACT = '/content/fins'
shutil.rmtree(EXTRACT, ignore_errors=True)
os.makedirs(EXTRACT, exist_ok=True)
with zipfile.ZipFile(ZIP_PATH) as z:
    z.extractall(EXTRACT)

# locate a tree that has both pde/ and experiments/
def find_root(base):
    for d, subs, _ in os.walk(base):
        if 'pde' in subs and 'experiments' in subs:
            return d
    return None

REPO = find_root(EXTRACT)

if REPO is None:
    # experiments-only archive: clone the repository and graft them in
    exp_src = None
    for d, subs, files_ in os.walk(EXTRACT):
        if os.path.basename(d) == 'experiments' and 'exp_common.py' in files_:
            exp_src = d
            break
    assert exp_src, 'no experiments/ with exp_common.py found in the archive'
    print('experiments-only archive detected; cloning the repository...')
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/KimKitsurag1/fins_repo.git',
                    '/content/fins_repo'], check=True)
    REPO = '/content/fins_repo'
    shutil.rmtree(os.path.join(REPO, 'experiments'), ignore_errors=True)
    shutil.copytree(exp_src, os.path.join(REPO, 'experiments'))

EXPDIR = os.path.join(REPO, 'experiments')
os.makedirs(os.path.join(EXPDIR, 'results'), exist_ok=True)
print('repo       :', REPO)
print('experiments:', sorted(f for f in os.listdir(EXPDIR) if f.endswith('.py')))

## 3. Dependencies

`neuraloperator` is what the conference version's runs used. Without it the repository
falls back to its own `SimpleFNO2d` (the guard is in `shared_modules.py`), which
trains fine but is a different architecture — so numbers are not directly
comparable with that version. Leave the flag on unless the install conflicts with
the Colab torch build.

Do NOT `pip install -r requirements.txt` here: it pins `torch<2.7`, which
would try to downgrade Colab's torch. The code runs on modern torch (all
experiments in this package were validated on 2.13); only the missing
pieces are installed.

In [ ]:
INSTALL_NEURALOP = True

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'tqdm'], check=False)
if INSTALL_NEURALOP:
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                        'neuraloperator'], capture_output=True, text=True)
    print(r.stdout[-1500:] or 'neuraloperator install: quiet')
    if r.returncode:
        print('FAILED — falling back to SimpleFNO2d:\n', r.stderr[-1500:])

try:
    from neuralop.models import FNO
    print('neuralop available -> paper architecture')
except ImportError:
    print('neuralop absent -> SimpleFNO2d fallback (see note above)')

## 4. Runner

Streams script output live rather than buffering it to the end, so a long KdV
run does not look frozen.

In [ ]:
import time

def run(script, *args, timeout=None):
    """Run an experiment script from experiments/ with live output."""
    cmd = [sys.executable, '-u', script, *map(str, args)]
    print('$', ' '.join(cmd), '\n' + '-' * 78)
    t0 = time.time()
    p = subprocess.Popen(cmd, cwd=EXPDIR, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    try:
        for line in p.stdout:
            print(line, end='')
        p.wait(timeout=timeout)
    except KeyboardInterrupt:
        p.kill()
        raise
    print('-' * 78)
    print(f'exit {p.returncode} in {time.time() - t0:.0f} s')
    return p.returncode

## 5. Oracle experiments (CPU, no surrogate)

These need no GPU and no training: the solution operator is exact (heat) or a
high-resolution solver (Burgers, KdV), so every residual is free of model
error. Runtimes on a Colab CPU: exp1 heat seconds, Burgers ~1 min, KdV ~3 min;
exp2-exp4 under a minute each; exp8 ~4 min (the resolved KdV branch dominates).

In [ ]:
run('exp1_oracle_torus_audit.py', '--pde', 'heat')

In [ ]:
run('exp1_oracle_torus_audit.py', '--pde', 'burgers')

In [ ]:
# --validate-repo-kdv additionally cross-checks against PDESolver.solve_kdv
run('exp1_oracle_torus_audit.py', '--pde', 'kdv', '--validate-repo-kdv')

In [ ]:
run('exp2_null_model.py', '--trials', 20)

In [ ]:
run('exp3_domain_geometry.py')

In [ ]:
run('exp4_symmetry_breaking.py')

In [ ]:
run('exp8_seam_window.py')

In [ ]:
# null floors for Table 1, all three equations (~2 min)
run('exp20_null_floors.py', '--pde', 'all')

In [ ]:
# falsifiable short-horizon predictions (KdV wrap-around; Burgers projective)
run('exp12_short_horizon.py')

### exp13 — the Burgers projective generator of the conference version is wrong

Analytic result verified numerically: the listed v5 has phi = -(x + tu),
whose characteristic leaves the defect M[Q] = -2(u + x u_x) in the
linearised Burgers equation; the true symmetry has phi = x - tu. The cell
below proves it three ways (determining equation on a wide domain, defect
identity, windowed JVP on the torus). The corrected generator is in
`pde/ground_truth.py`.

In [ ]:
run('exp13_projective_correction.py')

In [ ]:
# eps^2 detection law on a genuinely nonlinear operator (forced Burgers)
run('exp14_nonlinear_breaking.py')

## 6. Surrogate experiments (GPU)

Each trains at least one `S_theta`. Start with `--smoke` to confirm the
pipeline runs end to end on this machine, then remove it for the real budgets.

In [ ]:
# mechanics check across all four, a couple of minutes total
for s in ['exp5_delta_sweep.py', 'exp6_trivial_subalgebra.py',
          'exp10_leak_vs_delta.py']:
    run(s, '--smoke')
run('exp7_receptive_field.py')

### exp5 — surrogate quality sweep

Budgets spanning delta from roughly 1e-1 to 1e-2. The stability lemma
predicts residual ~ delta^2 for exact symmetries, i.e. slope 2 in log-log.

Protocol notes. Each (budget, repeat) is trained from a fresh
`torch.manual_seed(seed + repeat)`; without that the budgets share one
evolving RNG stream and, since the trainer keeps the LAST-epoch model with
no early stopping, the delta curve comes out non-monotone noise.
`--repeats 2` reports medians. The 64:100 point is dropped (delta ~ 0.4 is
an undertrained regime, useless for the fit). Runtime on a T4: roughly
4-5 hours; to split across sessions, run the big budget separately with
`'--budgets', '1024:600'` and merge the CSVs.

The script also prints the per-sample sanity check beside the sample-0
broadcast, which shows the size of the broadcast bias.

In [ ]:
run('exp5_delta_sweep.py', '--pde', 'heat', '--repeats', 2,
    '--budgets', '128:150', '256:250', '512:400', '1024:600')

### exp6 — factoring out the trivial subalgebra

Full budget on heat. Prediction: without the penalty the deflated-norm
sequence supports rank 6 with a growing b-share across slots; with
`lambda_b = 30` the b-share collapses and the rank drops to 3, matching the
essential algebra of the periodic problem.

Budget: surrogate ~75 min (trained once, checkpointed to
`results/exp6_surrogate_*.pt` and reused on re-runs), then 6 deflation
slots per lambda; the first `slot 1` line prints its own timing — total
~= surrogate + 12 x t_slot for both lambdas. On a T4 expect roughly 3-5
hours per lambda including the surrogate. To split across sessions, run
one lambda per session.

The script screens `--n-starts 3` random initialisations per slot (4
epochs each) and continues the best. Disable with
`--n-starts 1`. Budget grows to ~17 min/slot (~4.6 h both lambdas incl.
the surrogate).

Crash safety: every finished slot is saved
immediately to `<outdir>/exp6_slots_*/slot_k.npz` and a restart resumes
from the first missing slot with slot-indexed seeding, so a resumed run
reproduces the uninterrupted one; worst-case loss is one slot (~15 min)
plus the surrogate if it had not finished. Set `USE_DRIVE = True` above so
`--outdir` lands on Drive and survives VM recycling — with the default
local `results/` a dead VM still loses everything.

In [ ]:
# same env-aware location as every other experiment's artefacts
exp6_out = os.environ.get('FINS_RESULTS_DIR', 'results')
run('exp6_trivial_subalgebra.py', '--lam', 0, '--outdir', exp6_out)

In [ ]:
# reuses the surrogate checkpoint and any finished slots from exp6_out
run('exp6_trivial_subalgebra.py', '--lam', 30, '--outdir', exp6_out)

In [ ]:
# what did each slot actually find? (seconds; reads the saved npz)
run('exp6_analyze.py')

### exp9 — locality of the surrogate Jacobian

`--full` uses the conference version's config. Watch two things: whether the learned r95
grows like sqrt(t) as the exact kernel does, and how much Jacobian mass sits
outside the exact 4-sigma cone at early times.

In [ ]:
run('exp9_surrogate_locality.py', '--full')

### exp10 — does the windowed test survive on a surrogate?

Five budgets, three numbers each on the same `S_theta`:
delta, Jacobian leak, and the seam-windowed residual. Falling residual with
falling delta means the windowed test transfers to the pipeline; a plateau
means the surrogate's missing domain of dependence is the binding constraint
and `S_theta` needs an explicit locality term.

In [ ]:
run('exp10_leak_vs_delta.py',
    '--budgets', '128:150', '256:300', '512:400', '1024:600', '1024:1200')

### Multi-seed oracle loop (for the journal tables)

All oracle scripts accept `--seed`; nonzero seeds write suffixed CSVs
(`*_s1.csv`, ...) so runs do not overwrite each other. Roughly 15-20
minutes per extra seed on the Colab CPU.

In [ ]:
MULTI_SEED = False
SEEDS = [1, 2]            # seed 0 = the runs above

if MULTI_SEED:
    for sd in SEEDS:
        for pde in ('heat', 'burgers', 'kdv'):
            run('exp1_oracle_torus_audit.py', '--pde', pde, '--seed', sd)
        for sc in ('exp3_domain_geometry.py', 'exp4_symmetry_breaking.py',
                   'exp8_seam_window.py', 'exp12_short_horizon.py',
                   'exp13_projective_correction.py',
                   'exp14_nonlinear_breaking.py'):
            run(sc, '--seed', sd)
    mirror_to_drive(os.path.join(EXPDIR, 'results'))
else:
    print('MULTI_SEED = False — skipped')

## 6b. Table 1-3 pipeline (run_single_pde)

This is the benchmark of the earlier conference version: FINS plus five
baselines on one PDE and one seed per invocation. Budgets are that version's (roughly 6-8 GPU hours
per PDE per seed at full scale; `--quick` is a ~30-minute smoke). It runs
from `pde/`, not from `experiments/`.

In [ ]:
PDEDIR = os.path.join(REPO, 'pde')

def run_pde(*args, timeout=None):
    cmd = [sys.executable, '-u', 'run_single_pde.py', *map(str, args)]
    print('$', ' '.join(cmd), '\n' + '-' * 78)
    p = subprocess.Popen(cmd, cwd=PDEDIR, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='')
    p.wait(timeout=timeout)
    print('-' * 78, f'exit {p.returncode}')
    return p.returncode

# smoke: one PDE, reduced budgets
run_pde('--pde', 'heat', '--quick', '--seed', 42,
        '--device', 'cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
RUN_FULL_PIPELINE = False
PIPELINE_SEEDS = [42]
PIPELINE_PDES = ['heat', 'burgers', 'kdv']

if RUN_FULL_PIPELINE:
    dev = 'cuda' if torch.cuda.is_available() else 'cpu'
    for sd in PIPELINE_SEEDS:
        for pde in PIPELINE_PDES:
            run_pde('--pde', pde, '--seed', sd, '--device', dev)
            # mirror after every (pde, seed): sessions die
            import glob as _g, shutil as _sh
            for f in _g.glob(os.path.join(PDEDIR, '*.json')) + \
                     _g.glob(os.path.join(PDEDIR, '*.pt')):
                _sh.copy(f, os.path.join(EXPDIR, 'results'))
            mirror_to_drive(os.path.join(EXPDIR, 'results'))
else:
    print('RUN_FULL_PIPELINE = False — skipped')

In [ ]:
# collect pipeline outputs (json + checkpoints) into results/ for download
import glob as _g, shutil as _sh
n = 0
for f in _g.glob(os.path.join(PDEDIR, '*.json')) + \
         _g.glob(os.path.join(PDEDIR, '*.pt')):
    _sh.copy(f, os.path.join(EXPDIR, 'results'))
    n += 1
print(f'copied {n} pipeline artefacts into results/')
mirror_to_drive(os.path.join(EXPDIR, 'results'))

### exp11 — delta_J probe (the stability-lemma variable)

Measures, per budget on the same surrogate: test error delta, Jacobian
error delta_J over random probes, amplification k, sample-averaged leak,
and the seam-windowed residual. The read-out is which of R/delta^2 and
R/delta_J^2 is stable across budgets.

Budget note: exp10 showed that on this single-scale dataset n_train <=
512 plateaus at delta ~ 0.22 regardless of epochs — those budgets are
wasted GPU time. Sweep epochs at n = 1024 instead to spread delta inside
the trainable regime (~2.6 h for the three points below).

In [ ]:
# quick small-grid trend (optional; 15-20 min)
run('exp11_delta_j_probe.py',
    '--budgets', '1024:150', '1024:300', '1024:600',
    '--tag', 'gpu')

In [ ]:
# paper grid + neuralop architecture (~2.5-3 h)
run('exp11_delta_j_probe.py', '--full-grid',
    '--budgets', '1024:300', '1024:600',
    '--tag', 'fullgrid')

### exp45 — neural symmetry generators in 2D (u_t = div(u grad u))

Point-field MLP generators trained by the Ky Fan loss of the transport pencil. The exact-operator runs (N1-N4)
were done on CPU; this section runs the learned-operator arm (N5): an FNO2d stepper per seed (512 trajectories of
data seed 6000 + s, 60 epochs, ~48 min on the Colab GPU), then FINS's polynomial class and the neural generators through its
tangent. Outputs go to `FINS_RESULTS_DIR/neural2d/` (checkpoint `fno2d_pme_s{s}.pt`, `fno_m{m}_s{s}.json/.pt/.npz`).

In [ ]:
# N5: FNO2d trained once per seed, then m = 6 neural generators through it (FNO training ~48 min per seed; loaded if present)
for s in (42, 43, 44):
    run('exp45_neural_2d.py', '--op', 'fno', '--train-fno-if-missing', '--seed', s, '--m', 6,
        '--save', f'neural2d/fno_m6_s{s}')

In [ ]:
# m = 8 through the same FNOs (rank test inside the learned span)
for s in (42, 43, 44):
    run('exp45_neural_2d.py', '--op', 'fno', '--seed', s, '--m', 8, '--save', f'neural2d/fno_m8_s{s}')

In [ ]:
# per-trajectory rollout errors of the saved FNOs on the FINS training, held-out and unseen sets
run('exp45_fno_errors.py', '--seeds', '42,43,44', '--sets', '4000:16,5000:8,8000:8')

In [ ]:
# V4L: closure-gated rank of the 1D learned-operator runs (protocol data and fresh seeds)
import glob
R = os.environ.get('FINS_RESULTS_DIR', 'results')
files = sorted(glob.glob(f'{R}/fins_*_s4[234].npz') + glob.glob(f'{R}/fresh/fins_*_s4[234].npz'))
run('exp44_closure_rank.py', '--files', *files)

### exp45 through the learned FNO2d: training with mu = 0.01 (MF1-MF3) and finite flows (XF1-XF5)

Needs the saved FNOs `neural2d/fno2d_pme_s{s}.pt` and the runs `fno_m6_s{s}.npz`, `fno_m8_s{s}.pt` under
`FINS_RESULTS_DIR`. Predictions were registered in README_EXPERIMENTS.md before these runs.

In [ ]:
# MF1-MF3: m = 8 networks through the FNO with the smaller absolute term (~10 min per seed on a GPU)
for s in (42, 43, 44):
    run('exp45_neural_2d.py', '--op', 'fno', '--seed', s, '--m', 8, '--mu', 0.01, '--save', f'neural2d/fno_m8_mu001_s{s}')

In [ ]:
# XF1-XF4: finite flows of the polynomial generators through the FNO (saves after every generator)
for s in (42, 43, 44):
    run('exp45_flows.py', '--op', 'fno', '--device', 'cuda', '--seed', s, '--save', f'neural2d/flows/fno_flows_s{s}')

In [ ]:
# XF5: finite flows of the FNO-trained network directions (mu = 1 and mu = 0.01)
for tag in ('m8', 'm8_mu001'):
    for s in (42, 43, 44):
        run('exp45_flows.py', '--op', 'fno', '--device', 'cuda', '--seed', s, '--only', 'network', '--net',
            f'neural2d/fno_{tag}_s{s}', '--save', f'neural2d/flows/fno_flows_net_{tag}_s{s}')

In [ ]:
# P5a/P5b: LIG with k generators on Burgers, seeds 43 and 44 (added to the existing baseline files)
R = os.environ.get('FINS_RESULTS_DIR', 'results')
for s in (43, 44):
    run('exp41_baselines.py', '--pde', 'burgers', '--seed', s, '--methods', 'LIG_k', '--lig-epochs', 10,
        '--out', f'{R}/baselines_burgers_s{s}.npz')

In [ ]:
# P5b per seed: the Burgers table for seeds 43 and 44 separately (rows "LIG, 5 generators")
R = os.environ.get('FINS_RESULTS_DIR', 'results')
for s in (43, 44):
    run('exp42_tables.py', '--dir', R, '--pdes', 'burgers', '--seeds', s, '--null-draws', 0)

### exp48 — noise and temporal thinning through learned operators: heat

The perfect-model arm and LieNLSD (`results/noise_thin/`), KdV's learned operators and heat's
clean condition (the protocol's CNOs) are run separately. This section retrains heat's CNO on the corrupted data, which needs a
GPU (about 10 h per training on two CPU cores), and runs FINS through it in exp48's configuration with
`exp48_learned.py`. Predictions (LC, LT1-LT2, LN1-LN4) were registered in
README_EXPERIMENTS.md before any learned-arm result.

Conditions in this order: **clean-rt** (retrained on clean data: the control), **n1e-4**, **t4**, **t8**, **t2**,
**n3e-4**, **n1e-3**, **n1e-2**; seeds 42-44, so 24 trainings, each as long as a protocol training. Checkpoints
`S_theta_heat_s{s}_g64x128_cno_smooth_L1_{cond}.pt` go to `FINS_RESULTS_DIR`: mount Drive in section 1, so that a
disconnect costs one training; rerunning the cell skips every finished condition and seed.

Outputs: `FINS_RESULTS_DIR/noise_thin_learned_runs/{cond}/fins_heat_s{s}.npz` and `logs/`. The last cell zips them
(npz and logs, no checkpoints) for download.


In [ ]:
# exp48 learned arm, heat: CNOs retrained on the corrupted training data, FINS through them (resumable)
run('exp48_learned.py', 'run', '--pdes', 'heat', '--conds', 'clean-rt,n1e-4,t4,t8,t2,n3e-4,n1e-3,n1e-2',
    '--threads', 2)


In [ ]:
# progress: finished outputs per condition and the tail of every log (safe to run while the loop is stopped)
import glob
R = os.environ.get('FINS_RESULTS_DIR', 'results')
LR = os.path.join(R if os.path.isabs(R) else os.path.join(EXPDIR, R), 'noise_thin_learned_runs')
for d in sorted(glob.glob(os.path.join(LR, '*'))):
    print(os.path.basename(d), sorted(os.path.basename(f) for f in glob.glob(os.path.join(d, 'fins_*.npz'))))
    for log in sorted(glob.glob(os.path.join(d, 'logs', '*.log'))):
        lines = open(log).read().splitlines()
        print('   ', os.path.basename(log), '|', lines[-1][:140] if lines else '')


In [ ]:
# zip the learned-arm outputs (npz + logs; checkpoints stay in FINS_RESULTS_DIR) for download
import shutil
from google.colab import files as colab_files
arc = shutil.make_archive('/content/fins_noise_thin_learned_heat', 'zip', LR)
print(arc, os.path.getsize(arc), 'bytes')
colab_files.download(arc)


## 7. Results

In [ ]:
import pandas as pd, glob

RES = os.path.join(EXPDIR, 'results')
for f in sorted(glob.glob(os.path.join(RES, '*.csv'))):
    print('=' * 78)
    print(os.path.basename(f))
    print('=' * 78)
    display(pd.read_csv(f))

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def have(name):
    p = os.path.join(RES, name)
    return pd.read_csv(p) if os.path.exists(p) else None

# --- exp3: residual vs domain geometry ---
d = have('exp3_domain_geometry.csv')
if d is not None:
    d = d[~d.generator.str.contains('uniform')]
    fig, ax = plt.subplots(figsize=(6, 4))
    for g, sub in d.groupby('generator'):
        ax.loglog(sub.L_over_w, sub.resid_v_norm.clip(lower=1e-33),
                  'o-', label=g)
    ax.axhline(1e-2, ls='--', c='k', lw=1, label='threshold 1e-2')
    ax.set_xlabel('L / w'); ax.set_ylabel('Rayleigh residual')
    ax.set_title('exp3 — heat, exact operator: geometry controls the rank')
    ax.legend(fontsize=8); plt.tight_layout(); plt.show()

# --- exp4: residual vs breaking strength ---
d = have('exp4_symmetry_breaking.csv')
if d is not None:
    fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
    for ax, (mode, sub) in zip(axes, d.groupby('mode')):
        for g, s in sub.groupby('generator'):
            ax.loglog(s.eps, s.resid_v_norm.clip(lower=1e-30), 'o-', label=g)
        e = np.array(sorted(sub.eps.unique()))
        ax.loglog(e, 1e-1 * (e / e.max()) ** 2, 'k:', lw=1,
                  label='slope 2 guide')
        ax.axhline(1e-2, ls='--', c='k', lw=1)
        ax.set_title(f"perturbation mode '{mode}'")
        ax.set_xlabel('eps'); ax.legend(fontsize=8)
    axes[0].set_ylabel('Rayleigh residual')
    plt.tight_layout(); plt.show()

# --- exp8: residual vs seam margin ---
d = have('exp8_seam_window.csv')
if d is not None:
    fig, ax = plt.subplots(figsize=(6.5, 4))
    for (pde, L), sub in d.groupby(['pde', 'L']):
        for g, s in sub.groupby('generator'):
            ax.semilogy(s.margin / float(L),
                        s.resid_v_norm.clip(lower=1e-33), 'o-',
                        label=f'{pde} L={float(L):.2f} {g}', lw=1, ms=3)
    ax.axhline(1e-2, ls='--', c='k', lw=1)
    ax.set_xlabel('margin m / L'); ax.set_ylabel('windowed residual')
    ax.set_title('exp8 — seam-windowed test: heat > Burgers > KdV')
    ax.legend(fontsize=6, ncol=2); plt.tight_layout(); plt.show()

# --- exp10: windowed residual vs surrogate quality ---
d = have('exp10_leak_vs_delta.csv')
if d is not None:
    sub = d[np.isclose(d.m_over_L, 0.05)]
    fig, ax = plt.subplots(figsize=(6, 4))
    for g, s in sub.groupby('generator'):
        s = s.sort_values('delta')
        ax.loglog(s.delta, s.windowed_residual, 'o-', label=g)
    dd = np.array(sorted(sub.delta.unique()))
    ax.loglog(dd, sub.windowed_residual.max() * (dd / dd.max()) ** 2,
              'k:', lw=1, label='slope 2 guide')
    ax.axhline(1e-2, ls='--', c='k', lw=1)
    ax.set_xlabel('surrogate test rel-L2 delta')
    ax.set_ylabel('windowed residual at m = 0.05 L')
    ax.set_title('exp10 — does the windowed test track surrogate quality?')
    ax.legend(fontsize=8); plt.tight_layout(); plt.show()

## 8. Download results

In [ ]:
import shutil
from google.colab import files as colab_files

out = '/content/fins_results'
shutil.make_archive(out, 'zip', RES)
print(out + '.zip', os.path.getsize(out + '.zip'), 'bytes')
colab_files.download(out + '.zip')